In [0]:
def test_failed_login_streak(spark):

    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, when, sum as _sum

    data = [
        ("U1", "2024-01-01 10:00:00", "fail"),
        ("U1", "2024-01-01 10:05:00", "fail"),
        ("U1", "2024-01-01 10:10:00", "fail"),
        ("U1", "2024-01-01 10:20:00", "success")
    ]

    columns = ["user_id", "event_time", "status"]

    df = spark.createDataFrame(data, columns)

    df = df.withColumn("is_fail", when(col("status") == "fail", 1).otherwise(0))

    window_spec = Window.partitionBy("user_id").orderBy("event_time")

    df = df.withColumn(
        "group_id",
        _sum(when(col("status") == "success", 1).otherwise(0)).over(window_spec)
    )

    result_df = df.filter(col("is_fail") == 1) \
        .groupBy("user_id", "group_id") \
        .agg(_sum("is_fail").alias("fail_count")) \
        .filter(col("fail_count") >= 3)

    result = {r["user_id"] for r in result_df.collect()}
    expected = {"U1"}

    try:
        assert result == expected
        print("✅ Test Passed")

    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

test_failed_login_streak(spark)